### 📈 The Mathematics of Diversification and Hedging 

##### ▶️ Related Quant Guild Videos:

- [The 5 Papers That Built Modern Quant Finance](https://youtu.be/ZwS1gMGegrM)

- [I Bet You've Never Found Alpha (and I Can Prove It)](https://youtu.be/UzTJHs3-eT0)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

##### Diversification is a Statistical Mechanism

When you diversify a portfolio you are trying to reduce volatility to improve risk-adjusted return or reduce volatility drag

 $$
 g = \mu - \frac{1}{2} \sigma^2
 $$

  $$
  S_{\pi} = \frac{\mu - r_f}{\sigma}
  $$

 The variance of a two-asset portfolio is:
 
 $$
 \mathrm{Var}(w_1 X_1 + w_2 X_2) = w_1^2 \mathrm{Var}(X_1) + w_2^2 \mathrm{Var}(X_2) + 2w_1 w_2\, \mathrm{Cov}(X_1, X_2)
 $$

 $$
 \text{Diversification:} \quad \mathrm{Var}_{\text{portfolio}} < w_1^2 \mathrm{Var}(X_1) + w_2^2 \mathrm{Var}(X_2) \quad \text{if and only if} \quad \mathrm{Cov}(X_1, X_2) \ne \sqrt{\mathrm{Var}(X_1)\mathrm{Var}(X_2)}
 $$
 
 **So as long as $\mathrm{Cov}(X_1, X_2) \ne 1$ (i.e., correlation $\rho \ne 1$), diversification mechanically reduces risk.**

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Simulate data ---
np.random.seed(42)

n_years = 5
trading_days_per_year = 252
n_days = n_years * trading_days_per_year
dt = 1 / trading_days_per_year

mu_a = 0.12
mu_b = 0.09
sigma_a = 0.22
sigma_b = 0.18

s0_a = 100
s0_b = 100
s0_portfolio = 100

dates = pd.bdate_range(start="2026-01-01", periods=n_days + 1)

# Independent shocks => uncorrelated GBM paths
z_a = np.random.normal(size=n_days)
z_b = np.random.normal(size=n_days)

prices_a = [s0_a]
prices_b = [s0_b]

for t in range(n_days):
    prices_a.append(
        prices_a[-1] * np.exp((mu_a - 0.5 * sigma_a**2) * dt + sigma_a * np.sqrt(dt) * z_a[t])
    )
    prices_b.append(
        prices_b[-1] * np.exp((mu_b - 0.5 * sigma_b**2) * dt + sigma_b * np.sqrt(dt) * z_b[t])
    )

df = pd.DataFrame({
    "date": dates,
    "Stock A": prices_a,
    "Stock B": prices_b
})

df["Return A"] = df["Stock A"].pct_change()
df["Return B"] = df["Stock B"].pct_change()

# 50/50 daily rebalanced portfolio
df["Portfolio Return"] = 0.5 * df["Return A"] + 0.5 * df["Return B"]
df["Portfolio"] = s0_portfolio * (1 + df["Portfolio Return"].fillna(0)).cumprod()

# Drop first row for returns, but keep full price series for path animation
returns_df = df.dropna().reset_index(drop=True)

# --- Styling ---
off_white = "#e0e0e0"
stock_a_color = "#00ff88"
stock_b_color = "#33aaff"
portfolio_color = "#ffaa33"
baseline_color = "#777777"

# --- Summary stats ---
latest_date = df["date"].iloc[-1]

# Calculate annualized Sharpe ratios for Stock A, Stock B, Portfolio
def annualized_sharpe(returns, risk_free=0):
    # Assuming risk_free is daily; if want annualized, compare with annual rate.
    mean = np.nanmean(returns)
    std = np.nanstd(returns)
    sharpe = (mean - risk_free) / std * np.sqrt(trading_days_per_year)
    return sharpe

sharpe_a = annualized_sharpe(returns_df["Return A"])
sharpe_b = annualized_sharpe(returns_df["Return B"])
sharpe_portfolio = annualized_sharpe(returns_df["Portfolio Return"])

realized_corr = returns_df["Return A"].corr(returns_df["Return B"])

# --- Figure ---
fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.58, 0.42],
    horizontal_spacing=0.10,
    subplot_titles=(
        "Simulated GBM Price Paths",
        "Pairwise Daily Returns"
    )
)

# Scatter reference lines
x_min = min(returns_df["Return A"].min(), returns_df["Return B"].min())
x_max = max(returns_df["Return A"].max(), returns_df["Return B"].max())
x_pad = (x_max - x_min) * 0.15

scatter_min = x_min - x_pad
scatter_max = x_max + x_pad

fig.add_trace(
    go.Scatter(
        x=[scatter_min, scatter_max],
        y=[0, 0],
        mode="lines",
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        name="Zero Return",
        hoverinfo="skip",
        showlegend=False
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=[0, 0],
        y=[scatter_min, scatter_max],
        mode="lines",
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        name="Zero Return",
        hoverinfo="skip",
        showlegend=False
    ),
    row=1,
    col=2
)

# Animated Stock A price path
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["Stock A"].iloc[0]],
        mode="lines",
        line=dict(color=stock_a_color, width=3),
        name="Stock A",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Stock A: %{y:.2f}<extra></extra>"
    ),
    row=1,
    col=1
)

# Animated Stock B price path
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["Stock B"].iloc[0]],
        mode="lines",
        line=dict(color=stock_b_color, width=3),
        name="Stock B",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Stock B: %{y:.2f}<extra></extra>"
    ),
    row=1,
    col=1
)

# Animated 50/50 portfolio path
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["Portfolio"].iloc[0]],
        mode="lines",
        line=dict(color=portfolio_color, width=3),
        name="50/50 Portfolio",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Portfolio: %{y:.2f}<extra></extra>"
    ),
    row=1,
    col=1
)

# Animated pairwise return scatter
fig.add_trace(
    go.Scatter(
        x=[returns_df["Return A"].iloc[0]],
        y=[returns_df["Return B"].iloc[0]],
        mode="markers",
        marker=dict(
            size=7,
            color=off_white,
            opacity=0.75,
            line=dict(width=0)
        ),
        name="Daily Return Pairs",
        hovertemplate=(
            "Date: %{customdata|%Y-%m-%d}<br>"
            "Stock A Return: %{x:.2%}<br>"
            "Stock B Return: %{y:.2%}<extra></extra>"
        ),
        customdata=[returns_df["date"].iloc[0]]
    ),
    row=1,
    col=2
)

# --- Animation frames ---
frames = []
slider_steps = []

# 5-trading-day stride keeps the HTML responsive.
# Set frame_stride = 1 for a frame on every trading day.
frame_stride = 5
frame_indices = list(range(1, len(df), frame_stride))
if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

for i in frame_indices:
    frame_name = f"f{i}"

    # Price series uses df through i
    price_slice = df.iloc[:i + 1]

    # Returns are one row shorter, so use returns through i - 1
    return_slice = returns_df.iloc[:i]

    frames.append(go.Frame(
        data=[
            go.Scatter(
                x=price_slice["date"],
                y=price_slice["Stock A"]
            ),
            go.Scatter(
                x=price_slice["date"],
                y=price_slice["Stock B"]
            ),
            go.Scatter(
                x=price_slice["date"],
                y=price_slice["Portfolio"]
            ),
            go.Scatter(
                x=return_slice["Return A"],
                y=return_slice["Return B"],
                customdata=return_slice["date"]
            )
        ],
        traces=[2, 3, 4, 5],
        name=frame_name
    ))

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": df["date"].iloc[i].strftime("%Y"),
        "method": "animate"
    })

fig.frames = frames

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# Price axis range
price_min = min(df["Stock A"].min(), df["Stock B"].min(), df["Portfolio"].min())
price_max = max(df["Stock A"].max(), df["Stock B"].max(), df["Portfolio"].max())
price_pad = (price_max - price_min) * 0.08

# --- Layout ---
fig.update_layout(
    title=dict(
        text=(
            "Two Uncorrelated GBM Stock Paths and 50/50 Portfolio"
            f"<br><sup>5-Year Simulation Through {latest_date:%Y-%m-%d} · "
            f"Stock A Sharpe: {sharpe_a:.2f} · "
            f"Stock B Sharpe: {sharpe_b:.2f} · "
            f"Portfolio Sharpe: {sharpe_portfolio:.2f} · "
            f"Realized Return Corr: {realized_corr:.2f}</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1300,
    margin=dict(t=110, b=150, r=50, l=75),
    legend=dict(
        x=0.98,
        y=0.96,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white)
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": 20, "redraw": False},
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

# Subplot title styling
fig.update_annotations(font=dict(color=off_white, size=14))

# Left subplot: price paths
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=[price_min - price_pad, price_max + price_pad],
    title_text="Simulated Price / Portfolio Value"
)

# Right subplot: pairwise returns
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[scatter_min, scatter_max],
    title_text="Stock A Daily Return",
    tickformat=".1%"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=[scatter_min, scatter_max],
    title_text="Stock B Daily Return",
    tickformat=".1%"
)

fig.show()

###### ______________________________________________________________________________________________________________________________________

Problem: the statistical mechanism you are using fails exactly when you are trying to do that.
 
  $$
    \operatorname{Cov}(X, Y) = \mathbb{E}\left[(X - \mathbb{E}[X])(Y - \mathbb{E}[Y])\right], \quad
    \operatorname{Corr}(X, Y) = \frac{\operatorname{Cov}(X, Y)}{\sigma_X \sigma_Y} = \rho
  $$
 
  $$
    \operatorname{Corr}(X, Y) = \frac{\sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^n (x_i - \bar{x})^2} \sqrt{\sum_{i=1}^n (y_i - \bar{y})^2}}
    \;\;\not\!\!\longrightarrow\;\; \rho
  $$
  (The sample correlation does not necessarily converge to the true population correlation $\rho$)

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Simulate data ---
np.random.seed(42)

n_years = 5
trading_days_per_year = 252
n_days = n_years * trading_days_per_year
dt = 1 / trading_days_per_year

mu_a = 0.12
mu_b = 0.09
sigma_a = 0.22
sigma_b = 0.18

s0_a = 100
s0_b = 100
s0_portfolio = 100

dates = pd.bdate_range(start="2026-01-01", periods=n_days + 1)

# --- Crisis setup ---
crisis_start = pd.Timestamp("2028-06-01")
crisis_end = pd.Timestamp("2028-09-30")

crisis_mask = (dates[1:] >= crisis_start) & (dates[1:] <= crisis_end)
crisis_dates = dates[1:][crisis_mask]

# --- Normal GBM shocks: independent outside crisis ---
z_a = np.random.normal(size=n_days)
z_b = np.random.normal(size=n_days)

log_returns_a = (mu_a - 0.5 * sigma_a**2) * dt + sigma_a * np.sqrt(dt) * z_a
log_returns_b = (mu_b - 0.5 * sigma_b**2) * dt + sigma_b * np.sqrt(dt) * z_b

# --- Crisis regime: correlation goes to 1 and returns are negative but less severe ---
# Adjusted for less severe negative returns
n_crisis_days = crisis_mask.sum()

crisis_simple_returns = -np.clip(
    np.abs(np.random.normal(loc=0.007, scale=0.005, size=n_crisis_days)),
    0.001,   # was 0.003
    0.025    # was 0.060
)

crisis_log_returns = np.log1p(crisis_simple_returns)

log_returns_a[crisis_mask] = crisis_log_returns
log_returns_b[crisis_mask] = crisis_log_returns

# --- Build price paths ---
prices_a = s0_a * np.exp(np.r_[0, np.cumsum(log_returns_a)])
prices_b = s0_b * np.exp(np.r_[0, np.cumsum(log_returns_b)])

df = pd.DataFrame({
    "date": dates,
    "Stock A": prices_a,
    "Stock B": prices_b
})

df["Return A"] = df["Stock A"].pct_change()
df["Return B"] = df["Stock B"].pct_change()

# 50/50 daily rebalanced portfolio
df["Portfolio Return"] = 0.5 * df["Return A"] + 0.5 * df["Return B"]
df["Portfolio"] = s0_portfolio * (1 + df["Portfolio Return"].fillna(0)).cumprod()

df["Crisis"] = (df["date"] >= crisis_start) & (df["date"] <= crisis_end)

# Drop first row for returns, but keep full price series for path animation
returns_df = df.dropna().reset_index(drop=True)

# --- Styling ---
off_white = "#e0e0e0"
stock_a_color = "#00ff88"
stock_b_color = "#33aaff"
portfolio_color = "#ffaa33"
baseline_color = "#777777"
crisis_color = "#ff3333"

# --- Summary stats ---
latest_date = df["date"].iloc[-1]

latest_a = df["Stock A"].iloc[-1]
latest_b = df["Stock B"].iloc[-1]
latest_portfolio = df["Portfolio"].iloc[-1]

years_elapsed = n_days / trading_days_per_year

cagr_a = (latest_a / s0_a) ** (1 / years_elapsed) - 1
cagr_b = (latest_b / s0_b) ** (1 / years_elapsed) - 1
cagr_portfolio = (latest_portfolio / s0_portfolio) ** (1 / years_elapsed) - 1

realized_corr = returns_df["Return A"].corr(returns_df["Return B"])

crisis_returns_df = returns_df[returns_df["Crisis"]]
crisis_corr = crisis_returns_df["Return A"].corr(crisis_returns_df["Return B"])

# --- Figure ---
fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.58, 0.42],
    horizontal_spacing=0.10,
    subplot_titles=(
        "Simulated GBM Price Paths",
        "Pairwise Daily Returns"
    )
)

# Scatter reference lines
x_min = min(returns_df["Return A"].min(), returns_df["Return B"].min())
x_max = max(returns_df["Return A"].max(), returns_df["Return B"].max())
x_pad = (x_max - x_min) * 0.15

scatter_min = x_min - x_pad
scatter_max = x_max + x_pad

fig.add_trace(
    go.Scatter(
        x=[scatter_min, scatter_max],
        y=[0, 0],
        mode="lines",
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        name="Zero Return",
        hoverinfo="skip",
        showlegend=False
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=[0, 0],
        y=[scatter_min, scatter_max],
        mode="lines",
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        name="Zero Return",
        hoverinfo="skip",
        showlegend=False
    ),
    row=1,
    col=2
)

# 45-degree line to make crisis correlation = 1 visually obvious
fig.add_trace(
    go.Scatter(
        x=[scatter_min, scatter_max],
        y=[scatter_min, scatter_max],
        mode="lines",
        line=dict(color=crisis_color, width=1, dash="dot"),
        opacity=0.45,
        name="Corr = 1 Line",
        hoverinfo="skip",
        showlegend=False
    ),
    row=1,
    col=2
)

# Animated Stock A price path
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["Stock A"].iloc[0]],
        mode="lines",
        line=dict(color=stock_a_color, width=3),
        name="Stock A",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Stock A: %{y:.2f}<extra></extra>"
    ),
    row=1,
    col=1
)

# Animated Stock B price path
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["Stock B"].iloc[0]],
        mode="lines",
        line=dict(color=stock_b_color, width=3),
        name="Stock B",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Stock B: %{y:.2f}<extra></extra>"
    ),
    row=1,
    col=1
)

# Animated 50/50 portfolio path
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["Portfolio"].iloc[0]],
        mode="lines",
        line=dict(color=portfolio_color, width=3),
        name="50/50 Portfolio",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Portfolio: %{y:.2f}<extra></extra>"
    ),
    row=1,
    col=1
)

# Animated normal pairwise return scatter
fig.add_trace(
    go.Scatter(
        x=[returns_df["Return A"].iloc[0]],
        y=[returns_df["Return B"].iloc[0]],
        mode="markers",
        marker=dict(
            size=7,
            color=off_white,
            opacity=0.65,
            line=dict(width=0)
        ),
        name="Normal Return Pairs",
        hovertemplate=(
            "Date: %{customdata|%Y-%m-%d}<br>"
            "Stock A Return: %{x:.2%}<br>"
            "Stock B Return: %{y:.2%}<extra></extra>"
        ),
        customdata=[returns_df["date"].iloc[0]]
    ),
    row=1,
    col=2
)

# Animated crisis pairwise return scatter
fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode="markers",
        marker=dict(
            size=8,
            color=crisis_color,
            opacity=0.90,
            line=dict(width=0)
        ),
        name="2028 Crisis Return Pairs",
        hovertemplate=(
            "Date: %{customdata|%Y-%m-%d}<br>"
            "Stock A Return: %{x:.2%}<br>"
            "Stock B Return: %{y:.2%}<br>"
            "Crisis Corr ≈ 1<extra></extra>"
        ),
        customdata=[]
    ),
    row=1,
    col=2
)

# Crisis shading on the price-path chart
fig.add_vrect(
    x0=crisis_start,
    x1=crisis_end,
    fillcolor=crisis_color,
    opacity=0.18,
    line_width=0,
    layer="below",
    row=1,
    col=1
)

fig.add_annotation(
    x=crisis_start + (crisis_end - crisis_start) / 2,
    y=1.06,
    xref="x",
    yref="paper",
    text="2028 Crisis: correlations go to 1",
    showarrow=False,
    font=dict(color=crisis_color, size=13),
    bgcolor="rgba(30,30,30,0.75)",
    bordercolor="rgba(255,51,51,0.45)",
    borderwidth=1
)

# --- Animation frames ---
frames = []
slider_steps = []

# 5-trading-day stride keeps the HTML responsive.
# Set frame_stride = 1 for a frame on every trading day.
frame_stride = 5
frame_indices = list(range(1, len(df), frame_stride))
if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

for i in frame_indices:
    frame_name = f"f{i}"

    # Price series uses df through i
    price_slice = df.iloc[:i + 1]

    # Returns are one row shorter, so use returns through i - 1
    return_slice = returns_df.iloc[:i]

    normal_slice = return_slice[~return_slice["Crisis"]]
    crisis_slice = return_slice[return_slice["Crisis"]]

    frames.append(go.Frame(
        data=[
            go.Scatter(
                x=price_slice["date"],
                y=price_slice["Stock A"]
            ),
            go.Scatter(
                x=price_slice["date"],
                y=price_slice["Stock B"]
            ),
            go.Scatter(
                x=price_slice["date"],
                y=price_slice["Portfolio"]
            ),
            go.Scatter(
                x=normal_slice["Return A"],
                y=normal_slice["Return B"],
                customdata=normal_slice["date"]
            ),
            go.Scatter(
                x=crisis_slice["Return A"],
                y=crisis_slice["Return B"],
                customdata=crisis_slice["date"]
            )
        ],
        traces=[3, 4, 5, 6, 7],
        name=frame_name
    ))

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": df["date"].iloc[i].strftime("%Y"),
        "method": "animate"
    })

fig.frames = frames

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# Price axis range
price_min = min(df["Stock A"].min(), df["Stock B"].min(), df["Portfolio"].min())
price_max = max(df["Stock A"].max(), df["Stock B"].max(), df["Portfolio"].max())
price_pad = (price_max - price_min) * 0.08

# --- Layout ---
fig.update_layout(
    title=dict(
        text=(
            "Two GBM Stock Paths, 50/50 Portfolio, and Crisis Correlation Spike"
            f"<br><sup>5-Year Simulation Through {latest_date:%Y-%m-%d} · "
            f"Stock A CAGR: {cagr_a:.1%} · "
            f"Stock B CAGR: {cagr_b:.1%} · "
            f"Portfolio CAGR: {cagr_portfolio:.1%} · "
            f"Full-Sample Corr: {realized_corr:.2f} · "
            f"2028 Crisis Corr: {crisis_corr:.2f}</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1300,
    margin=dict(t=120, b=150, r=50, l=75),
    legend=dict(
        x=0.98,
        y=0.96,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white)
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": 20, "redraw": False},
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

# Subplot title styling
fig.update_annotations(font=dict(color=off_white, size=14))

# Left subplot: price paths
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=[price_min - price_pad, price_max + price_pad],
    title_text="Simulated Price / Portfolio Value"
)

# Right subplot: pairwise returns
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[scatter_min, scatter_max],
    title_text="Stock A Daily Return",
    tickformat=".1%"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=[scatter_min, scatter_max],
    title_text="Stock B Daily Return",
    tickformat=".1%"
)

fig.show()

###### ______________________________________________________________________________________________________________________________________

##### Real Diversification is Structural, not just Statistical

Structural implies statistical (physical implies stochastic independence), not the other way around

I don't need a backtest to tell you if a trading or investment strategy is good, as risk allocators we are effectively baseball players

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Simulate data ---
np.random.seed(42)

n_years = 5
trading_days_per_year = 252
n_days = n_years * trading_days_per_year
dt = 1 / trading_days_per_year

# Stock A: traditional risky asset
mu_a = 0.12
sigma_a = 0.22

# Sports MM Algo: lower-vol, positive-drift strategy
mu_algo = 0.18
sigma_algo = 0.08

s0_a = 100
s0_algo = 100
s0_portfolio = 100

dates = pd.bdate_range(start="2026-01-01", periods=n_days + 1)

# --- Crisis setup ---
crisis_start = pd.Timestamp("2028-06-01")
crisis_end = pd.Timestamp("2028-09-30")

crisis_mask = (dates[1:] >= crisis_start) & (dates[1:] <= crisis_end)
n_crisis_days = crisis_mask.sum()

# --- Normal shocks: independent outside crisis ---
z_a = np.random.normal(size=n_days)
z_algo = np.random.normal(size=n_days)

log_returns_a = (mu_a - 0.5 * sigma_a**2) * dt + sigma_a * np.sqrt(dt) * z_a
log_returns_algo = (mu_algo - 0.5 * sigma_algo**2) * dt + sigma_algo * np.sqrt(dt) * z_algo

# --- Crisis regime ---
# Stock A crashes: forced negative daily returns.
crisis_simple_returns_a = -np.clip(
    np.abs(np.random.normal(loc=0.015, scale=0.010, size=n_crisis_days)),
    0.003,
    0.060
)

log_returns_a[crisis_mask] = np.log1p(crisis_simple_returns_a)

# Sports MM Algo continues normally during crisis.
# This keeps the crisis-period pairwise return cloud uncorrelated instead of collapsing to corr = 1.
crisis_simple_returns_algo = np.random.normal(
    loc=mu_algo / trading_days_per_year,
    scale=sigma_algo / np.sqrt(trading_days_per_year),
    size=n_crisis_days
)

# Light floor avoids extreme simulated one-day losses, while still allowing normal-looking variation.
crisis_simple_returns_algo = np.clip(crisis_simple_returns_algo, -0.006, 0.014)

log_returns_algo[crisis_mask] = np.log1p(crisis_simple_returns_algo)

# --- Build price paths ---
prices_a = s0_a * np.exp(np.r_[0, np.cumsum(log_returns_a)])
prices_algo = s0_algo * np.exp(np.r_[0, np.cumsum(log_returns_algo)])

df = pd.DataFrame({
    "date": dates,
    "Stock A": prices_a,
    "Sports MM Algo": prices_algo
})

df["Return A"] = df["Stock A"].pct_change()
df["Return Algo"] = df["Sports MM Algo"].pct_change()

# 50/50 daily rebalanced portfolio
df["Portfolio Return"] = 0.5 * df["Return A"] + 0.5 * df["Return Algo"]
df["Portfolio"] = s0_portfolio * (1 + df["Portfolio Return"].fillna(0)).cumprod()

df["Crisis"] = (df["date"] >= crisis_start) & (df["date"] <= crisis_end)

# Drop first row for returns, but keep full price series for path animation
returns_df = df.dropna().reset_index(drop=True)

# --- Styling ---
off_white = "#e0e0e0"
stock_a_color = "#00ff88"
algo_color = "#ffd84d"
portfolio_color = "#ffaa33"
baseline_color = "#777777"
crisis_color = "#ff3333"
crisis_point_color = "#ffd84d"

# --- Summary stats ---
latest_date = df["date"].iloc[-1]

latest_a = df["Stock A"].iloc[-1]
latest_algo = df["Sports MM Algo"].iloc[-1]
latest_portfolio = df["Portfolio"].iloc[-1]

years_elapsed = n_days / trading_days_per_year

cagr_a = (latest_a / s0_a) ** (1 / years_elapsed) - 1
cagr_algo = (latest_algo / s0_algo) ** (1 / years_elapsed) - 1
cagr_portfolio = (latest_portfolio / s0_portfolio) ** (1 / years_elapsed) - 1

realized_corr = returns_df["Return A"].corr(returns_df["Return Algo"])

crisis_returns_df = returns_df[returns_df["Crisis"]]
crisis_corr = crisis_returns_df["Return A"].corr(crisis_returns_df["Return Algo"])

# --- Figure ---
fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.58, 0.42],
    horizontal_spacing=0.10,
    subplot_titles=(
        "Simulated Price Paths",
        "Pairwise Daily Returns"
    )
)

# Scatter reference lines
x_min = min(returns_df["Return A"].min(), returns_df["Return Algo"].min())
x_max = max(returns_df["Return A"].max(), returns_df["Return Algo"].max())
x_pad = (x_max - x_min) * 0.15

scatter_min = x_min - x_pad
scatter_max = x_max + x_pad

fig.add_trace(
    go.Scatter(
        x=[scatter_min, scatter_max],
        y=[0, 0],
        mode="lines",
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        name="Zero Return",
        hoverinfo="skip",
        showlegend=False
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=[0, 0],
        y=[scatter_min, scatter_max],
        mode="lines",
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        name="Zero Return",
        hoverinfo="skip",
        showlegend=False
    ),
    row=1,
    col=2
)

# Animated Stock A price path
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["Stock A"].iloc[0]],
        mode="lines",
        line=dict(color=stock_a_color, width=3),
        name="Stock A",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Stock A: %{y:.2f}<extra></extra>"
    ),
    row=1,
    col=1
)

# Animated Sports MM Algo path
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["Sports MM Algo"].iloc[0]],
        mode="lines",
        line=dict(color=algo_color, width=3),
        name="Sports MM Algo",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Sports MM Algo: %{y:.2f}<extra></extra>"
    ),
    row=1,
    col=1
)

# Animated 50/50 portfolio path
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["Portfolio"].iloc[0]],
        mode="lines",
        line=dict(color=portfolio_color, width=3),
        name="50/50 Portfolio",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Portfolio: %{y:.2f}<extra></extra>"
    ),
    row=1,
    col=1
)

# Animated normal pairwise return scatter
fig.add_trace(
    go.Scatter(
        x=[returns_df["Return A"].iloc[0]],
        y=[returns_df["Return Algo"].iloc[0]],
        mode="markers",
        marker=dict(
            size=7,
            color=off_white,
            opacity=0.65,
            line=dict(width=0)
        ),
        name="Normal Return Pairs",
        hovertemplate=(
            "Date: %{customdata|%Y-%m-%d}<br>"
            "Stock A Return: %{x:.2%}<br>"
            "Sports MM Algo Return: %{y:.2%}<extra></extra>"
        ),
        customdata=[returns_df["date"].iloc[0]]
    ),
    row=1,
    col=2
)

# Animated crisis pairwise return scatter
fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode="markers",
        marker=dict(
            size=8,
            color=crisis_point_color,
            opacity=0.95,
            line=dict(width=0)
        ),
        name="2028 Crisis Return Pairs",
        hovertemplate=(
            "Date: %{customdata|%Y-%m-%d}<br>"
            "Stock A Return: %{x:.2%}<br>"
            "Sports MM Algo Return: %{y:.2%}<br>"
            "Crisis returns remain uncorrelated<extra></extra>"
        ),
        customdata=[]
    ),
    row=1,
    col=2
)

# Crisis shading on the price-path chart
fig.add_vrect(
    x0=crisis_start,
    x1=crisis_end,
    fillcolor=crisis_color,
    opacity=0.18,
    line_width=0,
    layer="below",
    row=1,
    col=1
)

fig.add_annotation(
    x=crisis_start + (crisis_end - crisis_start) / 2,
    y=1.06,
    xref="x",
    yref="paper",
    text="2028 Crisis: Stock A sells off, Sports MM Algo keeps compounding",
    showarrow=False,
    font=dict(color=crisis_color, size=13),
    bgcolor="rgba(30,30,30,0.75)",
    bordercolor="rgba(255,51,51,0.45)",
    borderwidth=1
)

# --- Animation frames ---
frames = []
slider_steps = []

# 5-trading-day stride keeps the HTML responsive.
# Set frame_stride = 1 for a frame on every trading day.
frame_stride = 5
frame_indices = list(range(1, len(df), frame_stride))
if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

for i in frame_indices:
    frame_name = f"f{i}"

    # Price series uses df through i
    price_slice = df.iloc[:i + 1]

    # Returns are one row shorter, so use returns through i - 1
    return_slice = returns_df.iloc[:i]

    normal_slice = return_slice[~return_slice["Crisis"]]
    crisis_slice = return_slice[return_slice["Crisis"]]

    frames.append(go.Frame(
        data=[
            go.Scatter(
                x=price_slice["date"],
                y=price_slice["Stock A"]
            ),
            go.Scatter(
                x=price_slice["date"],
                y=price_slice["Sports MM Algo"]
            ),
            go.Scatter(
                x=price_slice["date"],
                y=price_slice["Portfolio"]
            ),
            go.Scatter(
                x=normal_slice["Return A"],
                y=normal_slice["Return Algo"],
                customdata=normal_slice["date"]
            ),
            go.Scatter(
                x=crisis_slice["Return A"],
                y=crisis_slice["Return Algo"],
                customdata=crisis_slice["date"]
            )
        ],
        traces=[2, 3, 4, 5, 6],
        name=frame_name
    ))

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": df["date"].iloc[i].strftime("%Y"),
        "method": "animate"
    })

fig.frames = frames

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# Price axis range
price_min = min(df["Stock A"].min(), df["Sports MM Algo"].min(), df["Portfolio"].min())
price_max = max(df["Stock A"].max(), df["Sports MM Algo"].max(), df["Portfolio"].max())
price_pad = (price_max - price_min) * 0.08

# --- Layout ---
fig.update_layout(
    title=dict(
        text=(
            "Stock A vs Sports MM Algo: Crisis Resilience and Uncorrelated Returns"
            f"<br><sup>5-Year Simulation Through {latest_date:%Y-%m-%d} · "
            f"Stock A CAGR: {cagr_a:.1%} · "
            f"Sports MM Algo CAGR: {cagr_algo:.1%} · "
            f"Portfolio CAGR: {cagr_portfolio:.1%} · "
            f"Full-Sample Corr: {realized_corr:.2f} · "
            f"2028 Crisis Corr: {crisis_corr:.2f}</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1300,
    margin=dict(t=120, b=150, r=50, l=75),
    legend=dict(
        x=0.98,
        y=0.96,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white)
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": 20, "redraw": False},
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

# Subplot title styling
fig.update_annotations(font=dict(color=off_white, size=14))

# Left subplot: price paths
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=[price_min - price_pad, price_max + price_pad],
    title_text="Simulated Price / Portfolio Value"
)

# Right subplot: pairwise returns
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[scatter_min, scatter_max],
    title_text="Stock A Daily Return",
    tickformat=".1%"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=[scatter_min, scatter_max],
    title_text="Sports MM Algo Daily Return",
    tickformat=".1%"
)

fig.show()

---

##### Hedging is a Contract that Resolves in the Future with Certainty

Depending on the contract, there may or may not be a transaction today *(i.e. options v. futures)*

$$
\text{Option payoff:} \quad (S_T - K)^+ \qquad \text{Forward/Future payoff:} \quad S_T - K
$$


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from math import erf, sqrt, log, exp

# --- Simulate data ---
np.random.seed(42)

n_years = 1
trading_days_per_year = 252
n_days = n_years * trading_days_per_year
dt = 1 / trading_days_per_year

# NVDA assumptions
s0_nvda = 210
entry_price = 180
put_strike = 180
shares = 1200

# Options convention: 1 listed option contract controls 100 shares
contract_multiplier = 100
put_contracts = shares / contract_multiplier

# GBM assumptions
mu_nvda = 0.18
sigma_nvda = 0.45

# Black-Scholes assumptions for marking the put option over time
risk_free_rate = 0.045
option_implied_vol = 0.45

dates = pd.bdate_range(start="2026-01-01", periods=n_days + 1)

# --- Helper functions ---
def norm_cdf(x):
    return 0.5 * (1 + erf(x / sqrt(2)))

def black_scholes_put_price(S, K, T, r, sigma):
    """
    European put option value per share.
    At expiration, value equals intrinsic value: max(K - S, 0).
    """
    if T <= 0:
        return max(K - S, 0)

    d1 = (log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)

    put_price = K * exp(-r * T) * norm_cdf(-d2) - S * norm_cdf(-d1)
    return put_price

# --- NVDA price simulation ---
z = np.random.normal(size=n_days)

log_returns_nvda = (
    (mu_nvda - 0.5 * sigma_nvda**2) * dt
    + sigma_nvda * np.sqrt(dt) * z
)

prices_nvda = s0_nvda * np.exp(np.r_[0, np.cumsum(log_returns_nvda)])

df = pd.DataFrame({
    "date": dates,
    "NVDA Price": prices_nvda
})

df["NVDA Return"] = df["NVDA Price"].pct_change()

# --- Protective put mechanics ---
# You are long 1,200 NVDA shares and buy puts with strike $180.
# The put creates downside protection below $180 while keeping upside participation.
df["Time Remaining"] = np.linspace(n_years, 0, len(df))

df["Put Value Per Share"] = [
    black_scholes_put_price(
        S=row["NVDA Price"],
        K=put_strike,
        T=row["Time Remaining"],
        r=risk_free_rate,
        sigma=option_implied_vol
    )
    for _, row in df.iterrows()
]

df["Put Position Value"] = shares * df["Put Value Per Share"]
df["Stock Position Value"] = shares * df["NVDA Price"]
df["Protected Position Value"] = df["Stock Position Value"] + df["Put Position Value"]

initial_put_value = df["Put Position Value"].iloc[0]
initial_protected_value = df["Protected Position Value"].iloc[0]

# At expiration, the put payoff offsets losses below the strike.
expiration_date = df["date"].iloc[-1]
expiration_stock_price = df["NVDA Price"].iloc[-1]
expiration_stock_value = shares * expiration_stock_price
expiration_put_value = shares * max(put_strike - expiration_stock_price, 0)
expiration_protected_value = expiration_stock_value + expiration_put_value

unhedged_gain_from_entry = shares * (expiration_stock_price - entry_price)
protected_floor_value = shares * put_strike
gross_protected_gain_from_entry = expiration_protected_value - shares * entry_price
net_protected_value_after_premium = expiration_protected_value - initial_put_value

# --- Styling ---
off_white = "#e0e0e0"
nvda_color = "#00ff88"
put_color = "#ffaa33"
strike_color = "#33aaff"
baseline_color = "#777777"

# --- Figure ---
fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.58, 0.42],
    horizontal_spacing=0.10,
    specs=[[{}, {"secondary_y": True}]],
    subplot_titles=(
        "Simulated NVDA Price Path",
        "Protective Put Option Value"
    )
)

# --- Left subplot: NVDA price path ---
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["NVDA Price"].iloc[0]],
        mode="lines",
        line=dict(color=nvda_color, width=3),
        name="NVDA Price",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>NVDA Price: $%{y:.2f}<extra></extra>"
    ),
    row=1,
    col=1
)

# Entry price line
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0], df["date"].iloc[-1]],
        y=[entry_price, entry_price],
        mode="lines",
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        name=f"Entry Price · ${entry_price:.0f}",
        hoverinfo="skip"
    ),
    row=1,
    col=1
)

# Put strike line
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0], df["date"].iloc[-1]],
        y=[put_strike, put_strike],
        mode="lines",
        line=dict(color=strike_color, width=1.5, dash="dot"),
        opacity=0.85,
        name=f"Put Strike · ${put_strike:.0f}",
        hoverinfo="skip"
    ),
    row=1,
    col=1
)

# --- Right subplot: put option value ---
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["Put Position Value"].iloc[0]],
        mode="lines",
        line=dict(color=put_color, width=3),
        name="Put Position Value",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Put Position Value: $%{y:,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=2,
    secondary_y=False
)

# Zero option value line
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0], df["date"].iloc[-1]],
        y=[0, 0],
        mode="lines",
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        name="Zero Option Value",
        hoverinfo="skip",
        showlegend=False
    ),
    row=1,
    col=2,
    secondary_y=False
)

# Put strike hline on secondary y-axis
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0], df["date"].iloc[-1]],
        y=[put_strike, put_strike],
        mode="lines",
        line=dict(color=strike_color, width=1.5, dash="dot"),
        opacity=0.85,
        name=f"Put Strike · ${put_strike:.0f}",
        hovertemplate="Put Strike: $%{y:.2f}<extra></extra>",
        showlegend=False
    ),
    row=1,
    col=2,
    secondary_y=True
)

# --- Animation frames ---
frames = []
slider_steps = []

frame_stride = 5
frame_indices = list(range(1, len(df), frame_stride))
if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

for i in frame_indices:
    frame_name = f"f{i}"
    path_slice = df.iloc[:i + 1]

    frames.append(go.Frame(
        data=[
            go.Scatter(
                x=path_slice["date"],
                y=path_slice["NVDA Price"]
            ),
            go.Scatter(
                x=path_slice["date"],
                y=path_slice["Put Position Value"]
            )
        ],
        traces=[0, 3],
        name=frame_name
    ))

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": df["date"].iloc[i].strftime("%b"),
        "method": "animate"
    })

fig.frames = frames

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# Axis ranges
price_min = min(df["NVDA Price"].min(), entry_price, put_strike)
price_max = max(df["NVDA Price"].max(), entry_price, put_strike)
price_pad = (price_max - price_min) * 0.10

put_min = df["Put Position Value"].min()
put_max = df["Put Position Value"].max()
put_pad = max((put_max - put_min) * 0.15, 1)

strike_axis_min = min(entry_price, put_strike, df["NVDA Price"].min())
strike_axis_max = max(entry_price, put_strike, df["NVDA Price"].max())
strike_axis_pad = (strike_axis_max - strike_axis_min) * 0.10

# --- Layout ---
fig.update_layout(
    title=dict(
        text=(
            "Protective Put on NVDA: Keeping Upside While Defining Downside"
            f"<br><sup>1,200 shares · Entry: ${entry_price:.0f} · "
            f"Spot Start: ${s0_nvda:.0f} · Put Strike: ${put_strike:.0f} · "
            f"Initial Put Cost: ${initial_put_value:,.0f} · "
            f"Expiration Spot: ${expiration_stock_price:.2f} · "
            f"Stock Value: ${expiration_stock_value:,.0f} · "
            f"Put Value: ${expiration_put_value:,.0f} · "
            f"Protected Value: ${expiration_protected_value:,.0f}</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1300,
    margin=dict(t=125, b=150, r=70, l=75),
    legend=dict(
        x=0.98,
        y=0.96,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white)
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": 20, "redraw": False},
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

# Subplot title styling
fig.update_annotations(font=dict(color=off_white, size=14))

# Left subplot axes
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=[price_min - price_pad, price_max + price_pad],
    title_text="NVDA Price",
    tickprefix="$"
)

# Right subplot axes
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=[max(0, put_min - put_pad), put_max + put_pad],
    title_text="Protective Put Position Value",
    tickprefix="$",
    secondary_y=False
)

fig.update_yaxes(
    showgrid=False,
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
    range=[strike_axis_min - strike_axis_pad, strike_axis_max + strike_axis_pad],
    title_text="Put Strike",
    tickprefix="$",
    secondary_y=True,
    row=1,
    col=2
)

fig.show()

###### ______________________________________________________________________________________________________________________________________

#### Hedging (Done Correctly) is Structural Diversification



 Statistical Correlation between Stock and Put Payoff:
 
 The statistical correlation measures how the returns of the underlying stock relate to the returns (payoff) from a put option.

 Given a time series of stock prices $S_t$ and put option payoffs $P_t$ at each time $t$, we compute the Pearson correlation as:

   $$
   \rho_{S,P} = \frac{\operatorname{Cov}(S_t,\,P_t)}{\sigma_{S} \,\sigma_{P}}
   $$

 where $\operatorname{Cov}$ is the covariance and $\sigma$ denotes standard deviation.
 Typically, the put payoff moves inversely to the stock, resulting in a negative correlation.


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from math import erf, sqrt, log, exp

# --- Simulate data ---
np.random.seed(42)

n_years = 1
trading_days_per_year = 252
n_days = n_years * trading_days_per_year
dt = 1 / trading_days_per_year

# NVDA assumptions
s0_nvda = 210
entry_price = 180
put_strike = 180
shares = 1200

# Options convention: 1 listed option contract controls 100 shares
contract_multiplier = 100
put_contracts = shares / contract_multiplier

# GBM assumptions
mu_nvda = 0.18
sigma_nvda = 0.45

# Black-Scholes assumptions for marking the put option over time
risk_free_rate = 0.045
base_option_implied_vol = 0.45
crisis_option_implied_vol = 0.90

dates = pd.bdate_range(start="2026-01-01", periods=n_days + 1)

# --- Crisis setup ---
crisis_start = pd.Timestamp("2026-06-01")
crisis_end = pd.Timestamp("2026-08-31")

crisis_mask_returns = (dates[1:] >= crisis_start) & (dates[1:] <= crisis_end)
n_crisis_days = crisis_mask_returns.sum()

# --- Helper functions ---
def norm_cdf(x):
    return 0.5 * (1 + erf(x / sqrt(2)))

def black_scholes_put_price(S, K, T, r, sigma):
    """
    European put option value per share.
    At expiration, value equals intrinsic value: max(K - S, 0).
    """
    if T <= 0:
        return max(K - S, 0)

    d1 = (log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)

    put_price = K * exp(-r * T) * norm_cdf(-d2) - S * norm_cdf(-d1)
    return put_price

# --- NVDA price simulation ---
z = np.random.normal(size=n_days)

log_returns_nvda = (
    (mu_nvda - 0.5 * sigma_nvda**2) * dt
    + sigma_nvda * np.sqrt(dt) * z
)

# --- Crisis regime: force NVDA into a drawdown ---
crisis_simple_returns_nvda = -np.clip(
    np.abs(np.random.normal(loc=0.018, scale=0.012, size=n_crisis_days)),
    0.004,
    0.070
)

log_returns_nvda[crisis_mask_returns] = np.log1p(crisis_simple_returns_nvda)

prices_nvda = s0_nvda * np.exp(np.r_[0, np.cumsum(log_returns_nvda)])

df = pd.DataFrame({
    "date": dates,
    "NVDA Price": prices_nvda
})

df["Crisis"] = (df["date"] >= crisis_start) & (df["date"] <= crisis_end)
df["NVDA Return"] = df["NVDA Price"].pct_change()

# --- Protective put mechanics ---
# You are long 1,200 NVDA shares and buy puts with strike $180.
# The put gains value when NVDA falls, reducing downside volatility in the combined position.
df["Time Remaining"] = np.linspace(n_years, 0, len(df))

df["Option IV"] = np.where(
    df["Crisis"],
    crisis_option_implied_vol,
    base_option_implied_vol
)

df["Put Value Per Share"] = [
    black_scholes_put_price(
        S=row["NVDA Price"],
        K=put_strike,
        T=row["Time Remaining"],
        r=risk_free_rate,
        sigma=row["Option IV"]
    )
    for _, row in df.iterrows()
]

df["Put Position Value"] = shares * df["Put Value Per Share"]
df["Stock Position Value"] = shares * df["NVDA Price"]
df["Protected Portfolio Value"] = df["Stock Position Value"] + df["Put Position Value"]

df["Put Return"] = df["Put Position Value"].pct_change()
df["Protected Portfolio Return"] = df["Protected Portfolio Value"].pct_change()

# Clean return series for correlation/scatter
returns_df = df.dropna(subset=[
    "NVDA Return",
    "Put Return",
    "Protected Portfolio Return"
]).copy()

returns_df = returns_df.replace([np.inf, -np.inf], np.nan).dropna(
    subset=["NVDA Return", "Put Return", "Protected Portfolio Return"]
).reset_index(drop=True)

# --- Summary stats ---
initial_put_value = df["Put Position Value"].iloc[0]
initial_stock_value = df["Stock Position Value"].iloc[0]
initial_protected_value = df["Protected Portfolio Value"].iloc[0]

expiration_date = df["date"].iloc[-1]
expiration_stock_price = df["NVDA Price"].iloc[-1]
expiration_stock_value = df["Stock Position Value"].iloc[-1]
expiration_put_value = shares * max(put_strike - expiration_stock_price, 0)
expiration_protected_value = expiration_stock_value + expiration_put_value

stock_ann_vol = returns_df["NVDA Return"].std() * np.sqrt(trading_days_per_year)
put_ann_vol = returns_df["Put Return"].std() * np.sqrt(trading_days_per_year)
protected_ann_vol = returns_df["Protected Portfolio Return"].std() * np.sqrt(trading_days_per_year)

full_sample_corr = returns_df["NVDA Return"].corr(returns_df["Put Return"])

crisis_returns_df = returns_df[returns_df["Crisis"]]
crisis_corr = crisis_returns_df["NVDA Return"].corr(crisis_returns_df["Put Return"])

# --- Styling ---
off_white = "#e0e0e0"
nvda_color = "#00ff88"
put_color = "#ffaa33"
portfolio_color = "#33aaff"
strike_color = "#33aaff"
baseline_color = "#777777"
crisis_color = "#ff3333"
crisis_point_color = "#ffd84d"

# --- Figure ---
fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.58, 0.42],
    horizontal_spacing=0.10,
    subplot_titles=(
        "NVDA Stock, Protective Put, and Protected Portfolio",
        "NVDA Returns vs Put Returns"
    )
)

# --- Left subplot: portfolio values ---
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["Stock Position Value"].iloc[0]],
        mode="lines",
        line=dict(color=nvda_color, width=3),
        name="NVDA Stock Value",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "NVDA Stock Value: $%{y:,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["Put Position Value"].iloc[0]],
        mode="lines",
        line=dict(color=put_color, width=3),
        name="Put Position Value",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Put Position Value: $%{y:,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["Protected Portfolio Value"].iloc[0]],
        mode="lines",
        line=dict(color=portfolio_color, width=3),
        name="Protected Portfolio Value",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Protected Portfolio Value: $%{y:,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# Put floor line: 1,200 shares protected at $180, before considering option premium
protected_floor_value = shares * put_strike

fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0], df["date"].iloc[-1]],
        y=[protected_floor_value, protected_floor_value],
        mode="lines",
        line=dict(color=strike_color, width=1.5, dash="dot"),
        opacity=0.85,
        name=f"Put Floor · ${protected_floor_value:,.0f}",
        hoverinfo="skip"
    ),
    row=1,
    col=1
)

# Crisis shading
fig.add_vrect(
    x0=crisis_start,
    x1=crisis_end,
    fillcolor=crisis_color,
    opacity=0.18,
    line_width=0,
    layer="below",
    row=1,
    col=1
)

fig.add_annotation(
    x=crisis_start + (crisis_end - crisis_start) / 2,
    y=1.06,
    xref="x",
    yref="paper",
    text="Crash: NVDA falls, put value rises, portfolio volatility is reduced",
    showarrow=False,
    font=dict(color=crisis_color, size=13),
    bgcolor="rgba(30,30,30,0.75)",
    bordercolor="rgba(255,51,51,0.45)",
    borderwidth=1
)

# --- Right subplot: NVDA return vs put return scatter ---
x_min = returns_df["NVDA Return"].min()
x_max = returns_df["NVDA Return"].max()
y_min = returns_df["Put Return"].quantile(0.01)
y_max = returns_df["Put Return"].quantile(0.99)

x_pad = (x_max - x_min) * 0.15
y_pad = (y_max - y_min) * 0.15

scatter_x_min = x_min - x_pad
scatter_x_max = x_max + x_pad
scatter_y_min = y_min - y_pad
scatter_y_max = y_max + y_pad

# Zero return lines
fig.add_trace(
    go.Scatter(
        x=[scatter_x_min, scatter_x_max],
        y=[0, 0],
        mode="lines",
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        hoverinfo="skip",
        showlegend=False
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=[0, 0],
        y=[scatter_y_min, scatter_y_max],
        mode="lines",
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        hoverinfo="skip",
        showlegend=False
    ),
    row=1,
    col=2
)

# Negative relationship guide line
fig.add_trace(
    go.Scatter(
        x=[scatter_x_min, scatter_x_max],
        y=[scatter_y_max, scatter_y_min],
        mode="lines",
        line=dict(color=put_color, width=1.5, dash="dot"),
        opacity=0.70,
        name="Negative Relationship",
        hoverinfo="skip",
        showlegend=False
    ),
    row=1,
    col=2
)

# Animated normal return pairs
fig.add_trace(
    go.Scatter(
        x=[returns_df["NVDA Return"].iloc[0]],
        y=[returns_df["Put Return"].iloc[0]],
        mode="markers",
        marker=dict(
            size=7,
            color=off_white,
            opacity=0.65,
            line=dict(width=0)
        ),
        name="Normal Return Pairs",
        hovertemplate=(
            "Date: %{customdata|%Y-%m-%d}<br>"
            "NVDA Return: %{x:.2%}<br>"
            "Put Return: %{y:.2%}<extra></extra>"
        ),
        customdata=[returns_df["date"].iloc[0]]
    ),
    row=1,
    col=2
)

# Animated crisis return pairs
fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode="markers",
        marker=dict(
            size=8,
            color=crisis_point_color,
            opacity=0.95,
            line=dict(width=0)
        ),
        name="Crash Return Pairs",
        hovertemplate=(
            "Date: %{customdata|%Y-%m-%d}<br>"
            "NVDA Return: %{x:.2%}<br>"
            "Put Return: %{y:.2%}<br>"
            "Put rises when NVDA falls<extra></extra>"
        ),
        customdata=[]
    ),
    row=1,
    col=2
)

# --- Animation frames ---
frames = []
slider_steps = []

frame_stride = 5
frame_indices = list(range(1, len(df), frame_stride))
if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

for i in frame_indices:
    frame_name = f"f{i}"

    value_slice = df.iloc[:i + 1]

    current_date = df["date"].iloc[i]
    return_slice = returns_df[returns_df["date"] <= current_date]

    normal_slice = return_slice[~return_slice["Crisis"]]
    crisis_slice = return_slice[return_slice["Crisis"]]

    frames.append(go.Frame(
        data=[
            go.Scatter(
                x=value_slice["date"],
                y=value_slice["Stock Position Value"]
            ),
            go.Scatter(
                x=value_slice["date"],
                y=value_slice["Put Position Value"]
            ),
            go.Scatter(
                x=value_slice["date"],
                y=value_slice["Protected Portfolio Value"]
            ),
            go.Scatter(
                x=normal_slice["NVDA Return"],
                y=normal_slice["Put Return"],
                customdata=normal_slice["date"]
            ),
            go.Scatter(
                x=crisis_slice["NVDA Return"],
                y=crisis_slice["Put Return"],
                customdata=crisis_slice["date"]
            )
        ],
        traces=[0, 1, 2, 7, 8],
        name=frame_name
    ))

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": df["date"].iloc[i].strftime("%b"),
        "method": "animate"
    })

fig.frames = frames

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# Left axis range
value_min = min(
    df["Stock Position Value"].min(),
    df["Put Position Value"].min(),
    df["Protected Portfolio Value"].min(),
    protected_floor_value
)

value_max = max(
    df["Stock Position Value"].max(),
    df["Put Position Value"].max(),
    df["Protected Portfolio Value"].max(),
    protected_floor_value
)

value_pad = (value_max - value_min) * 0.08

# --- Layout ---
fig.update_layout(
    title=dict(
        text=(
            "Protective Put on NVDA: Negative Correlation Reduces Portfolio Volatility During a Crash"
            f"<br><sup>1,200 shares · Spot Start: ${s0_nvda:.0f} · "
            f"Put Strike: ${put_strike:.0f} · Initial Put Cost: ${initial_put_value:,.0f} · "
            f"Expiration Spot: ${expiration_stock_price:.2f} · "
            f"Stock Value: ${expiration_stock_value:,.0f} · "
            f"Put Value: ${expiration_put_value:,.0f} · "
            f"Protected Value: ${expiration_protected_value:,.0f} · "
            f"Stock Vol: {stock_ann_vol:.1%} · "
            f"Protected Vol: {protected_ann_vol:.1%} · "
            f"Put Corr: {full_sample_corr:.2f} · "
            f"Crash Corr: {crisis_corr:.2f}</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1300,
    margin=dict(t=135, b=150, r=70, l=75),
    legend=dict(
        x=0.98,
        y=0.96,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white)
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": 20, "redraw": False},
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

# Subplot title styling
fig.update_annotations(font=dict(color=off_white, size=14))

# Left subplot axes
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=[value_min - value_pad, value_max + value_pad],
    title_text="Dollar Value",
    tickprefix="$"
)

# Right subplot axes
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[scatter_x_min, scatter_x_max],
    title_text="NVDA Daily Return",
    tickformat=".1%"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=[scatter_y_min, scatter_y_max],
    title_text="Put Daily Return",
    tickformat=".1%"
)

fig.show()

###### ______________________________________________________________________________________________________________________________________

##### Incorrect v.s. Correct Diversification on Long Term Portfolio Growth

In the long run, even simple strategy signficantly outpreforms standard diversification (think SPY) in the context of CAGR

The strategy is simnple in design, difficult to implement in practice

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Simulate data ---
np.random.seed(42)

n_years = 10
trading_days_per_year = 252
n_days = n_years * trading_days_per_year
dt = 1 / trading_days_per_year

# Portfolio assumptions
initial_portfolio_value = 100
s0_spy = 500

# SPY assumptions
mu_spy = 0.085
sigma_spy = 0.19

# Risk-free rate for Sharpe calculation
risk_free_rate = 0.03

dates = pd.bdate_range(start="2026-01-01", periods=n_days + 1)

# --- Crisis setup ---
# Repeated market stress events create volatility drag for the unhedged portfolio.
crisis_windows = [
    ("2028-03-01", "2028-10-31"),
    ("2031-04-01", "2031-10-31"),
    ("2034-02-01", "2034-08-31")
]

crisis_mask_returns = np.zeros(n_days, dtype=bool)

for start, end in crisis_windows:
    crisis_mask_returns |= (
        (dates[1:] >= pd.Timestamp(start)) &
        (dates[1:] <= pd.Timestamp(end))
    )

n_crisis_days = crisis_mask_returns.sum()

# --- SPY return simulation ---
z = np.random.normal(size=n_days)

spy_returns = (
    mu_spy * dt
    + sigma_spy * np.sqrt(dt) * z
)

# Crisis regime: negative-biased, high-volatility market stress.
crisis_returns = np.clip(
    np.random.normal(
        loc=-0.0028,
        scale=0.020,
        size=n_crisis_days
    ),
    -0.065,
    0.035
)

spy_returns[crisis_mask_returns] = crisis_returns

spy_prices = s0_spy * np.cumprod(np.r_[1, 1 + spy_returns])

df = pd.DataFrame({
    "date": dates,
    "SPY Price": spy_prices
})

df["SPY Return"] = df["SPY Price"].pct_change()
df["Crisis"] = False

for start, end in crisis_windows:
    df.loc[
        (df["date"] >= pd.Timestamp(start)) &
        (df["date"] <= pd.Timestamp(end)),
        "Crisis"
    ] = True

# --- Unhedged portfolio ---
df["Unhedged SPY Portfolio"] = initial_portfolio_value * (
    df["SPY Price"] / df["SPY Price"].iloc[0]
)

df["Unhedged Return"] = df["Unhedged SPY Portfolio"].pct_change()

# --- Rolling protective-put strategy ---
# This is a stylized teaching model:
# - The portfolio keeps long SPY exposure.
# - It pays an ongoing premium drag for protection.
# - During sharp drawdowns, the put overlay gains value.
# - Those gains are monetized and redeployed into the portfolio.
#
# The goal is to visually illustrate how convex downside protection can reduce
# volatility drag and improve long-run compounding through repeated stress events.

equity_beta = 0.85
cash_weight = 1 - equity_beta

annual_put_premium_drag = 0.025
daily_put_premium_drag = annual_put_premium_drag / trading_days_per_year

put_trigger = 0.008
put_convexity = 0.45

normal_put_monetization_multiplier = 0.10
crisis_put_monetization_multiplier = 1.00

# Put overlay gains when SPY return is meaningfully negative.
# Example: if SPY falls more than the trigger, the put starts contributing positively.
raw_put_overlay_return = put_convexity * np.maximum(
    -spy_returns - put_trigger,
    0
)

put_overlay_return = np.where(
    crisis_mask_returns,
    raw_put_overlay_return * crisis_put_monetization_multiplier,
    raw_put_overlay_return * normal_put_monetization_multiplier
)

rolling_put_returns = (
    equity_beta * spy_returns
    + cash_weight * (risk_free_rate / trading_days_per_year)
    - daily_put_premium_drag
    + put_overlay_return
)

df["Rolling Put Return"] = np.r_[np.nan, rolling_put_returns]

df["Rolling Put Portfolio"] = initial_portfolio_value * np.cumprod(
    np.r_[1, 1 + rolling_put_returns]
)

df["Put Overlay Contribution"] = np.r_[np.nan, put_overlay_return]
df["Put Premium Drag"] = -daily_put_premium_drag

# --- Return data ---
returns_df = df.dropna(subset=[
    "Unhedged Return",
    "Rolling Put Return"
]).reset_index(drop=True)

# --- Portfolio metrics ---
def portfolio_stats(value_series, return_series, years_elapsed):
    if len(return_series) < 30 or return_series.std() == 0:
        return {
            "CAGR": 0,
            "Annualized Vol": 0,
            "Arithmetic Return": 0,
            "Volatility Drag": 0,
            "Sharpe": 0
        }

    cagr = (value_series.iloc[-1] / value_series.iloc[0]) ** (1 / years_elapsed) - 1
    annualized_vol = return_series.std() * np.sqrt(trading_days_per_year)
    arithmetic_ann_return = return_series.mean() * trading_days_per_year
    volatility_drag = arithmetic_ann_return - cagr

    sharpe = (
        (return_series.mean() - risk_free_rate / trading_days_per_year)
        / return_series.std()
        * np.sqrt(trading_days_per_year)
    )

    return {
        "CAGR": cagr,
        "Annualized Vol": annualized_vol,
        "Arithmetic Return": arithmetic_ann_return,
        "Volatility Drag": volatility_drag,
        "Sharpe": sharpe
    }

def stats_through(i):
    """
    Compute metrics through index i so the bar chart can animate.
    """
    sliced = df.iloc[:i + 1].copy()
    sliced_returns = sliced.dropna(subset=["Unhedged Return", "Rolling Put Return"])

    years_elapsed = max(
        (sliced["date"].iloc[-1] - sliced["date"].iloc[0]).days / 365.25,
        1 / trading_days_per_year
    )

    unhedged_stats_now = portfolio_stats(
        sliced["Unhedged SPY Portfolio"],
        sliced_returns["Unhedged Return"],
        years_elapsed
    )

    rolling_put_stats_now = portfolio_stats(
        sliced["Rolling Put Portfolio"],
        sliced_returns["Rolling Put Return"],
        years_elapsed
    )

    return unhedged_stats_now, rolling_put_stats_now

# Final stats
unhedged_stats, rolling_put_stats = stats_through(len(df) - 1)

# --- Styling ---
off_white = "#e0e0e0"
unhedged_color = "#00ff88"
hedged_color = "#33aaff"
baseline_color = "#777777"
crisis_color = "#ff3333"

# --- Figure ---
fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.58, 0.42],
    horizontal_spacing=0.10,
    specs=[[{}, {"secondary_y": True}]],
    subplot_titles=(
        "10-Year Portfolio Paths",
        "Sharpe Ratio and Volatility Drag"
    )
)

# --- Left subplot: animated portfolio paths ---
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["Unhedged SPY Portfolio"].iloc[0]],
        mode="lines",
        line=dict(color=unhedged_color, width=3),
        name="Unhedged SPY",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Unhedged SPY Portfolio: %{y:.2f}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["Rolling Put Portfolio"].iloc[0]],
        mode="lines",
        line=dict(color=hedged_color, width=3),
        name="Rolling Protective Put",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Rolling Put Portfolio: %{y:.2f}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0], df["date"].iloc[-1]],
        y=[initial_portfolio_value, initial_portfolio_value],
        mode="lines",
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        name="Initial Capital",
        hoverinfo="skip",
        showlegend=False
    ),
    row=1,
    col=1
)

# Crisis shading
for start, end in crisis_windows:
    fig.add_vrect(
        x0=pd.Timestamp(start),
        x1=pd.Timestamp(end),
        fillcolor=crisis_color,
        opacity=0.15,
        line_width=0,
        layer="below",
        row=1,
        col=1
    )

fig.add_annotation(
    x=pd.Timestamp("2028-07-01"),
    y=1.06,
    xref="x",
    yref="paper",
    text="Stress events: puts are monetized and redeployed",
    showarrow=False,
    font=dict(color=crisis_color, size=13),
    bgcolor="rgba(30,30,30,0.75)",
    bordercolor="rgba(255,51,51,0.45)",
    borderwidth=1
)

# --- Right subplot: grouped metric bars ---
metric_labels = ["Sharpe Ratio", "Volatility Drag"]

initial_unhedged_stats, initial_rolling_put_stats = stats_through(21)

initial_unhedged_sharpe = initial_unhedged_stats["Sharpe"]
initial_hedged_sharpe = initial_rolling_put_stats["Sharpe"]

initial_unhedged_drag = initial_unhedged_stats["Volatility Drag"]
initial_hedged_drag = initial_rolling_put_stats["Volatility Drag"]

# Sharpe bars on primary y-axis
fig.add_trace(
    go.Bar(
        x=["Sharpe Ratio"],
        y=[initial_unhedged_sharpe],
        name="Unhedged SPY",
        marker_color=unhedged_color,
        opacity=0.90,
        text=[f"{initial_unhedged_sharpe:.2f}"],
        textposition="outside",
        offsetgroup="unhedged",
        hovertemplate="Unhedged SPY<br>Sharpe Ratio: %{y:.2f}<extra></extra>"
    ),
    row=1,
    col=2,
    secondary_y=False
)

fig.add_trace(
    go.Bar(
        x=["Sharpe Ratio"],
        y=[initial_hedged_sharpe],
        name="Rolling Protective Put",
        marker_color=hedged_color,
        opacity=0.90,
        text=[f"{initial_hedged_sharpe:.2f}"],
        textposition="outside",
        offsetgroup="hedged",
        hovertemplate="Rolling Protective Put<br>Sharpe Ratio: %{y:.2f}<extra></extra>"
    ),
    row=1,
    col=2,
    secondary_y=False
)

# Volatility drag bars on secondary y-axis
fig.add_trace(
    go.Bar(
        x=["Volatility Drag"],
        y=[initial_unhedged_drag],
        name="Unhedged SPY",
        marker_color=unhedged_color,
        opacity=0.65,
        text=[f"{initial_unhedged_drag:.1%}"],
        textposition="outside",
        offsetgroup="unhedged",
        showlegend=False,
        hovertemplate="Unhedged SPY<br>Volatility Drag: %{y:.2%}<extra></extra>"
    ),
    row=1,
    col=2,
    secondary_y=True
)

fig.add_trace(
    go.Bar(
        x=["Volatility Drag"],
        y=[initial_hedged_drag],
        name="Rolling Protective Put",
        marker_color=hedged_color,
        opacity=0.65,
        text=[f"{initial_hedged_drag:.1%}"],
        textposition="outside",
        offsetgroup="hedged",
        showlegend=False,
        hovertemplate="Rolling Protective Put<br>Volatility Drag: %{y:.2%}<extra></extra>"
    ),
    row=1,
    col=2,
    secondary_y=True
)

# --- Animation frames ---
frames = []
slider_steps = []

frame_stride = 21
frame_indices = list(range(21, len(df), frame_stride))
if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

# Precompute animated metric ranges so bars do not rescale wildly or disappear.
all_sharpe_values = []
all_drag_values = []

for i in frame_indices:
    u_stats_i, h_stats_i = stats_through(i)

    all_sharpe_values.extend([
        u_stats_i["Sharpe"],
        h_stats_i["Sharpe"]
    ])

    all_drag_values.extend([
        u_stats_i["Volatility Drag"],
        h_stats_i["Volatility Drag"]
    ])

for i in frame_indices:
    frame_name = f"f{i}"
    path_slice = df.iloc[:i + 1]

    u_stats_i, h_stats_i = stats_through(i)

    unhedged_sharpe_i = u_stats_i["Sharpe"]
    hedged_sharpe_i = h_stats_i["Sharpe"]

    unhedged_drag_i = u_stats_i["Volatility Drag"]
    hedged_drag_i = h_stats_i["Volatility Drag"]

    frames.append(go.Frame(
        data=[
            go.Scatter(
                x=path_slice["date"],
                y=path_slice["Unhedged SPY Portfolio"]
            ),
            go.Scatter(
                x=path_slice["date"],
                y=path_slice["Rolling Put Portfolio"]
            ),
            go.Bar(
                x=["Sharpe Ratio"],
                y=[unhedged_sharpe_i],
                text=[f"{unhedged_sharpe_i:.2f}"]
            ),
            go.Bar(
                x=["Sharpe Ratio"],
                y=[hedged_sharpe_i],
                text=[f"{hedged_sharpe_i:.2f}"]
            ),
            go.Bar(
                x=["Volatility Drag"],
                y=[unhedged_drag_i],
                text=[f"{unhedged_drag_i:.1%}"]
            ),
            go.Bar(
                x=["Volatility Drag"],
                y=[hedged_drag_i],
                text=[f"{hedged_drag_i:.1%}"]
            )
        ],
        # Include the bar traces in every frame so they persist and animate.
        traces=[0, 1, 3, 4, 5, 6],
        name=frame_name
    ))

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": True},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": df["date"].iloc[i].strftime("%Y"),
        "method": "animate"
    })

fig.frames = frames

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# --- Axis ranges ---
portfolio_min = min(
    df["Unhedged SPY Portfolio"].min(),
    df["Rolling Put Portfolio"].min(),
    initial_portfolio_value
)

portfolio_max = max(
    df["Unhedged SPY Portfolio"].max(),
    df["Rolling Put Portfolio"].max(),
    initial_portfolio_value
)

portfolio_pad = (portfolio_max - portfolio_min) * 0.08

# These are not used anymore, but retained if needed for consistency.
sharpe_min = min(all_sharpe_values)
sharpe_max = max(all_sharpe_values)
sharpe_pad = max((sharpe_max - sharpe_min) * 0.30, 0.15)

drag_min = min(all_drag_values)
drag_max = max(all_drag_values)
drag_pad = max((drag_max - drag_min) * 0.30, 0.005)

# --- Layout ---
fig.update_layout(
    title=dict(
        text=(
            "Rolling Protective Puts vs Unhedged SPY: Monetized Convexity Reduces Volatility Drag"
            f"<br><sup>10-Year Simulation · "
            f"Unhedged CAGR: {unhedged_stats['CAGR']:.1%} · "
            f"Rolling Put CAGR: {rolling_put_stats['CAGR']:.1%} · "
            f"Unhedged Vol: {unhedged_stats['Annualized Vol']:.1%} · "
            f"Rolling Put Vol: {rolling_put_stats['Annualized Vol']:.1%} · "
            f"Unhedged Sharpe: {unhedged_stats['Sharpe']:.2f} · "
            f"Rolling Put Sharpe: {rolling_put_stats['Sharpe']:.2f} · "
            f"Unhedged Vol Drag: {unhedged_stats['Volatility Drag']:.1%} · "
            f"Rolling Put Vol Drag: {rolling_put_stats['Volatility Drag']:.1%}</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1300,
    margin=dict(t=140, b=150, r=80, l=75),
    showlegend=False,  # REMOVE legend
    hovermode="closest",
    barmode="group",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": 45, "redraw": True},
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": True},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

# Subplot title styling
fig.update_annotations(font=dict(color=off_white, size=14))

# Left subplot axes
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=[portfolio_min - portfolio_pad, portfolio_max + portfolio_pad],
    title_text="Portfolio Value, Indexed to 100"
)

# Right subplot axes
fig.update_xaxes(
    showgrid=False,
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
    row=1,
    col=2,
    title_text="Metric",
    categoryorder="array",
    categoryarray=metric_labels
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
    row=1,
    col=2,
    secondary_y=False,
    title_text="Sharpe Ratio",
    range=[-1, sharpe_max + sharpe_pad]  # set minimum to -1 for Sharpe
)

fig.update_yaxes(
    showgrid=False,
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
    row=1,
    col=2,
    secondary_y=True,
    title_text="Volatility Drag",
    tickformat=".1%",
    range=[-0.02, drag_max + drag_pad]  # set minimum to -0.02 for vol drag
)

fig.show()

---

#### 💭 Closing Thoughts and Future Topics

 **📑 TL;DW Executive Summary**

 **Diversification**
   - Diversification involves spreading investments across multiple uncorrelated (or less correlated) assets.
   - The main idea is that not all assets will react the same way to market events, so poor performance in one can be offset by better performance in others.
   - **Consequence:** This approach reduces the overall portfolio volatility, creating smoother returns over time but retains some market risk.

 **Hedging**
   - Hedging is the practice of deliberately taking an offsetting position in a specific asset or using financial instruments (like options or futures) to protects against particular risks.
   - The main idea is risk transfer or risk cancellation—directly offsetting known exposures rather than just diluting them.
   - **Consequence:** While effective hedging can directly guard against targeted downside moves, it may also reduce potential upside and can come with costs.

   - This lesson compared how diversification and hedging operate under different market scenarios and illustrated their distinct impacts on portfolio performance.
   - Using visual analysis, we saw that diversification often smooths returns over time, whereas effective hedging can directly protect against specific downside risks.

###### ______________________________________________________________________________________________________________________________________

 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System

---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$